In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

In [10]:
# step 1: load and clean data
df = pd.read_csv('Housing.csv')

# converting text yes/no to 1 and 0 so the model can read it
cols_to_map = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in cols_to_map:
    df[col] = df[col].map({'yes': 1, 'no': 0})

df['furnishingstatus'] = df['furnishingstatus'].map({'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0})
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,2
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,2
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,1
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,2
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,2


In [11]:
# separate features and target
X = df.drop('price', axis=1).values
y = df['price'].values

# scaling 
def normalize(X_data):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_data)
    return X_scaled

X_scaled = normalize(X)

In [12]:
print("---- RIDGE REGRESSION ----")

poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X_scaled)
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.2, random_state=42)

ridge = Ridge(alpha=10.0)
ridge.fit(X_train, y_train)
preds_ridge = ridge.predict(X_test)
print("MSE:", mean_squared_error(y_test, preds_ridge))
print("R2:", r2_score(y_test, preds_ridge))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_scores_ridge = []
for train_idx, test_idx in kf.split(X_poly):
    X_tr, X_te = X_poly[train_idx], X_poly[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    
    temp_model = Ridge(alpha=10.0)
    temp_model.fit(X_tr, y_tr)
    r2_scores_ridge.append(r2_score(y_te, temp_model.predict(X_te)))

print("Average K-Fold R2:", np.mean(r2_scores_ridge))

---- RIDGE REGRESSION ----
MSE: 1752935337200.8125
R2: 0.6531979472511507
Average K-Fold R2: 0.5788348188852914


In [13]:
print("\n---- LASSO REGRESSION ----")
lasso = Lasso(alpha=100.0)
lasso.fit(X_train, y_train)
preds_lasso = lasso.predict(X_test)
print("MSE:", mean_squared_error(y_test, preds_lasso))
print("R2:", r2_score(y_test, preds_lasso))

r2_scores_lasso = []
for train_idx, test_idx in kf.split(X_poly):
    X_tr, X_te = X_poly[train_idx], X_poly[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    
    temp_model = Lasso(alpha=100.0)
    temp_model.fit(X_tr, y_tr)
    r2_scores_lasso.append(r2_score(y_te, temp_model.predict(X_te)))

print("Average K-Fold R2:", np.mean(r2_scores_lasso))


---- LASSO REGRESSION ----
MSE: 1763729232266.0415
R2: 0.6510624748886877
Average K-Fold R2: 0.5440593905297002


c:\Tools\miniconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.681e+13, tolerance: 1.344e+11
  model = cd_fast.enet_coordinate_descent(
c:\Tools\miniconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.681e+13, tolerance: 1.344e+11
  model = cd_fast.enet_coordinate_descent(
c:\Tools\miniconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.666e+13, tolerance: 1.433e+11
  mod